In [ ]:
!python settings.py

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.autonotebook import tqdm

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

from settings import RERANKER_ID, OUTPUT_DIR, DEVICE, BATCH_SIZE

os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
data = {
    'corpus': pd.read_parquet('data/processed/corpus_data.parquet'),
    'train' : pd.read_parquet('data/processed/train_data.parquet'),
    'test'  : pd.read_parquet('data/processed/test_data.parquet')
}
for split in ['train', 'test']:
    data[split]['cid']          = data[split]['cid'].apply(lambda x: x.tolist())
    data[split]['context_list'] = data[split]['context_list'].apply(lambda x: x.tolist())

In [ ]:
fine_tuned_model = SentenceTransformer(OUTPUT_DIR, device=DEVICE)
reranker_model   = CrossEncoder(RERANKER_ID, device=DEVICE)

In [ ]:
passages          = data['corpus']['text'].tolist()
corpus_embeddings = fine_tuned_model.encode(
    passages, 
    batch_size=BATCH_SIZE,
    convert_to_numpy=True, 
    normalize_embeddings=True,
    show_progress_bar=True, 
    device=DEVICE
).astype(np.float32)

In [ ]:
d         = corpus_embeddings.shape[1]  # 768
cpu_index = faiss.IndexFlatIP(d)

res       = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
gpu_index.add(corpus_embeddings)

In [ ]:
final_cpu_index = faiss.index_gpu_to_cpu(gpu_index)
faiss.write_index(final_cpu_index, 'data/retrieval/legal_faiss.index')

In [ ]:
legal_index = faiss.read_index('data/retrieval/legal_faiss.index')

In [ ]:
def search_and_rerank(emb_model, rerank_model, query, index, faiss_k, final_k, batch_size=32):
    # FAISS
    q_emb = emb_model.encode(
        query, 
        convert_to_numpy=True, 
        normalize_embeddings=True,
    ).astype(np.float32).reshape(1, -1)
    
    scores, indices = index.search(q_emb, faiss_k) # shape: (1, faiss_k)
    
    cand_idxs   = indices[0]
    cand_scores = scores[0]
    cand_texts  = [passages[i] for i in cand_idxs]
    
    # Reranking
    pairs     = [(query, text) for text in cand_texts]
    re_scores = rerank_model.predict(pairs, batch_size=batch_size, convert_to_numpy=True)
    
    # Sort by rerank score
    merged = [{
        'index'         : int(cand_idxs[i]),
        'bm25_score'    : float(cand_scores[i]),   # dot-prod
        'rerank_score'  : float(re_scores[i]),
        'text'          : cand_texts[i]
    } for i in range(len(cand_idxs))]

    merged.sort(key=lambda x: x['rerank_score'], reverse=True)
    return merged[:final_k]

In [ ]:
query = 'Hợp đồng lao động là gì?'
hits  = search_rerank(emb_model, reranker, query, index,
                      faiss_k=50, final_k=10, batch_size=32)

for h in hits:
    print(f"[Rank {hits.index(h)+1}] rerank={h['rerank_score']:.4f}  emb={h['bm25_score']:.4f}\n{h['text']}\n{'-'*80}")

In [ ]:
# def search(model, query, index, k=10):
#     query_embedding = model.encode(
#         query, 
#         convert_to_numpy=True, 
#         normalize_embeddings=True,
#     ).astype(np.float32).reshape(1, -1)

#     scores, indices = index.search(query_embedding, k*3)
#     hits = [{'score': scores[0][i], 'index': indices[0][i]} for i in range(len(scores[0]))]
#     return hits

In [ ]:
# hits = search(
#     model=fine_tuned_model, 
#     query='Hợp đồng lao động là gì?', 
#     index=legal_index, 
#     k=10
# )

# for rank, hit in enumerate(hits):
#     print(f"[Rank: {rank + 1}]")
#     print(f"(Index: {hit['index']}Score: {hit['score']:.4f})\n")
#     print(passages[hit['index']])
#     print('-' * 100)
#     print()